In [2]:
# ==============================================================================
# POTATO DATA PREPROCESSING — ENVIRONMENT SETUP
# ==============================================================================

import os
import shutil
import numpy as np
from PIL import Image
import tensorflow as tf

print("=" * 75)
print("POTATO CNN — DATA PREPROCESSING")
print("=" * 75)

# ------------------------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------------------------
# 2. Project paths
# ------------------------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

RAW_DIR = os.path.join(
    PROJECT_DIR,
    "1st Raw_Data",
    "Potato"
)

PREPROCESSING_DIR = os.path.join(
    PROJECT_DIR,
    "3rd Preprocessing"
)

OUTPUT_FILE = os.path.join(
    PREPROCESSING_DIR,
    "potato_processed_data.npz"
)

# ------------------------------------------------------------------------------
# 3. Create preprocessing folder
# ------------------------------------------------------------------------------

os.makedirs(PREPROCESSING_DIR, exist_ok=True)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nRaw Potato dataset:")
print(RAW_DIR)

print("\nPreprocessing directory:")
print(PREPROCESSING_DIR)

print("\nOutput file:")
print(OUTPUT_FILE)

# ------------------------------------------------------------------------------
# 4. Verify raw dataset structure
# ------------------------------------------------------------------------------

print("\n" + "=" * 75)
print("VERIFYING POTATO RAW DATASET")
print("=" * 75)

required_paths = [
    os.path.join(RAW_DIR, "Internal_PlantVillage", "Healthy"),
    os.path.join(RAW_DIR, "Internal_PlantVillage", "Early_Blight"),
    os.path.join(RAW_DIR, "Internal_PlantVillage", "Late_Blight"),
    os.path.join(RAW_DIR, "External_Natural", "Healthy"),
    os.path.join(RAW_DIR, "External_Natural", "Early_Blight"),
    os.path.join(RAW_DIR, "External_Natural", "Late_Blight")
]

for path in required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"❌ Required dataset folder missing:\n{path}"
        )

print("✅ All 6 Potato dataset groups found.")

print("\n" + "=" * 75)
print("PREPROCESSING ENVIRONMENT READY")
print("=" * 75)

POTATO CNN — DATA PREPROCESSING
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Project directory:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)

Raw Potato dataset:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/1st Raw_Data/Potato

Preprocessing directory:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing

Output file:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/potato_processed_data.npz

VERIFYING POTATO RAW DATASET
✅ All 6 Potato dataset groups found.

PREPROCESSING ENVIRONMENT READY


In [ ]:
# ==============================================================================
# POTATO CNN — COMPLETE DATA PREPROCESSING
# ==============================================================================

import os
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf

print("=" * 75)
print("POTATO CNN — DATA PREPROCESSING")
print("=" * 75)

# ------------------------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------------------------

RAW_DIR = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/1st Raw_Data/Potato"

OUTPUT_DIR = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing"

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "potato_processed_data.npz"
)

IMG_SIZE = (224, 224)

class_names = [
    "Healthy",
    "Early_Blight",
    "Late_Blight"
]

print("\nRaw dataset:", RAW_DIR)
print("Output file :", OUTPUT_FILE)

# ------------------------------------------------------------------------------
# 2. LOAD + COMBINE INTERNAL AND EXTERNAL DATA
# ------------------------------------------------------------------------------

images = []
labels = []

source_counts = {}

print("\n" + "=" * 75)
print("LOADING + COMBINING POTATO DATA")
print("=" * 75)

for source in ["Internal_PlantVillage", "External_Natural"]:

    source_path = os.path.join(RAW_DIR, source)

    print(f"\n{source}")

    for class_index, class_name in enumerate(class_names):

        class_path = os.path.join(source_path, class_name)

        files = [
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
        ]

        valid_count = 0

        for file_name in files:

            file_path = os.path.join(class_path, file_name)

            try:
                image = Image.open(file_path).convert("RGB")

                # Resize
                image = image.resize(IMG_SIZE)

                # Convert to float32 + normalize 0–1
                image_array = np.array(
                    image,
                    dtype=np.float32
                ) / 255.0

                # Safety check
                if image_array.shape != (224, 224, 3):
                    continue

                if not np.isfinite(image_array).all():
                    continue

                images.append(image_array)
                labels.append(class_index)

                valid_count += 1

            except Exception:
                continue

        source_counts[(source, class_name)] = valid_count

        print(
            f"{class_name:15s}: "
            f"{valid_count:4d} valid images"
        )

# ------------------------------------------------------------------------------
# 3. CONVERT TO NUMPY ARRAYS
# ------------------------------------------------------------------------------

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.int32)

print("\n" + "=" * 75)
print("COMBINED DATASET")
print("=" * 75)

print("Total images :", len(X))
print("Image shape  :", X.shape)
print("Label shape  :", y.shape)

# ------------------------------------------------------------------------------
# 4. VERIFY COMBINED CLASS COUNTS
# ------------------------------------------------------------------------------

print("\nClass distribution:")

for i, class_name in enumerate(class_names):

    count = np.sum(y == i)

    print(
        f"{class_name:15s}: "
        f"{count:4d}"
    )

# ------------------------------------------------------------------------------
# 5. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ------------------------------------------------------------------------------

# First: 80% Train + 20% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
    shuffle=True
)

# Then: split temporary 50/50
# → 10% Validation
# → 10% Test

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
    shuffle=True
)

print("\n" + "=" * 75)
print("STRATIFIED DATA SPLIT")
print("=" * 75)

print("Training   :", X_train.shape[0])
print("Validation :", X_val.shape[0])
print("Testing    :", X_test.shape[0])

# ------------------------------------------------------------------------------
# 6. FINAL PREPROCESSING VERIFICATION
# ------------------------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL PREPROCESSING VERIFICATION")
print("=" * 75)

print("\nX_train")
print("dtype :", X_train.dtype)
print("min   :", X_train.min())
print("max   :", X_train.max())
print("shape :", X_train.shape)

print("\nX_val")
print("dtype :", X_val.dtype)
print("min   :", X_val.min())
print("max   :", X_val.max())
print("shape :", X_val.shape)

print("\nX_test")
print("dtype :", X_test.dtype)
print("min   :", X_test.min())
print("max   :", X_test.max())
print("shape :", X_test.shape)

# ------------------------------------------------------------------------------
# 7. VERIFY CLASS DISTRIBUTION AFTER SPLIT
# ------------------------------------------------------------------------------

print("\n" + "=" * 75)
print("CLASS DISTRIBUTION AFTER SPLIT")
print("=" * 75)

for i, class_name in enumerate(class_names):

    train_count = np.sum(y_train == i)
    val_count = np.sum(y_val == i)
    test_count = np.sum(y_test == i)

    print(
        f"{class_name:15s} | "
        f"Train: {train_count:4d} | "
        f"Val: {val_count:4d} | "
        f"Test: {test_count:4d}"
    )

# ------------------------------------------------------------------------------
# 8. SAVE PREPROCESSED DATA
# ------------------------------------------------------------------------------

np.savez_compressed(
    OUTPUT_FILE,

    X_train=X_train,
    y_train=y_train,

    X_val=X_val,
    y_val=y_val,

    X_test=X_test,
    y_test=y_test,

    class_names=np.array(class_names)
)

print("\n" + "=" * 75)
print("PREPROCESSING COMPLETE")
print("=" * 75)

print("✅ Internal + External data combined")
print("✅ Images cleaned and validated")
print("✅ Resized to 224 × 224 × 3")
print("✅ Normalized to 0–1")
print("✅ Stratified 80/10/10 split completed")
print("✅ Preprocessed dataset saved")

print("\nSaved file:")
print(OUTPUT_FILE)

print("=" * 75)

POTATO CNN — DATA PREPROCESSING

Raw dataset: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/1st Raw_Data/Potato
Output file : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/potato_processed_data.npz

LOADING + COMBINING POTATO DATA

Internal_PlantVillage
Healthy        :  152 valid images
Early_Blight   : 1000 valid images
Late_Blight    : 1000 valid images

External_Natural
Healthy        : 1000 valid images
Early_Blight   :  300 valid images
Late_Blight    : 1000 valid images

COMBINED DATASET
Total images : 4452
Image shape  : (4452, 224, 224, 3)
Label shape  : (4452,)

Class distribution:
Healthy        : 1152
Early_Blight   : 1300
Late_Blight    : 2000

STRATIFIED DATA SPLIT
Training   : 3561
Validation : 445
Testing    : 446

FINAL PREPROCESSING VERIFICATION

X_train
dtype : float32
min   : 0.0
max   : 1.0
shape : (3561, 224, 224, 3)

X_val
dtype : float32
min   : 0.0
max   : 1.0
shape : (445, 224, 224, 3)

X_test
dtype : flo